<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c05-header.png" alt="Nextia Learning · Machine Learning: From Problem to Reliable Model" width="100%">

# Solution: Choose the learning task

**[Choose the learning task](https://learning.nextia-ai.com/courses/ml/m01/choose-the-learning-task/)** · Machine Learning: From Problem to Reliable Model · Module 1, lesson 4 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** tell regression, classification and clustering apart on Larkfield's real columns, and to see when a rule, a lookup or a report is a better answer than a model.

| | |
|---|---|
| **Time** | About 30 minutes |
| **Needs** | An internet connection for the setup cells. No account. The first cell installs pandas 3.0.6 and scikit-learn 1.9.1. |
| **You should know** | Features and labels, regression and classification, from [Supervised learning](https://learning.nextia-ai.com/courses/ml/m01/supervised-learning/#features-and-labels); clustering, from [Unsupervised learning](https://learning.nextia-ai.com/courses/ml/m01/unsupervised-learning/). |
| **You do not need** | A local project. The setup below downloads the data. |
| **Tested** | Python 3.14.6 with pandas 3.0.6, scikit-learn 1.9.1 and matplotlib 3.11.2, in Jupyter on macOS, from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/ml/m01/choose-the-learning-task/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C05/M01-L04-choose-the-learning-task/choose-the-learning-task-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs pandas 3.0.6 and scikit-learn 1.9.1, the versions that the course uses, and matplotlib for the charts. It needs an internet connection and takes up to a minute the first time.

You should see `pandas 3.0.6` and `scikit-learn 1.9.1`. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. Restarting is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.0.6 scikit-learn==1.9.1 matplotlib
import pandas; print("pandas", pandas.__version__)
import sklearn; print("scikit-learn", sklearn.__version__)

## Setup: get the practice data

Run the next cell. It downloads the practice data of this course (about 2.5 MB) into a folder `ticket-model/data`, and checks each file's checksum. It is the same data that `get_data.py` downloads in [Define a baseline](https://learning.nextia-ai.com/courses/ml/m02/define-a-baseline/). It needs an internet connection and no account, and takes about 10 seconds.

You should see `Ready: train 16939 rows (1983 escalated), valid 1179 (183), test 1002 (157).` If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings (it needs a verified phone number).
- `wrong checksum`: delete the folder `ticket-model` (in Colab: the **Files** panel on the left), then run the cell again.

The data is made up for this course; it has no real people.

In [ ]:
# Setup: download the C05 data into ticket-model/data/ and check it.
# Standard library only: the same steps as get_data.py.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/C05/"
FILES = {
    "train.csv": "63b8732a28e891189cc28a9bcd505a093a9c4c51c01f3d0e1fba6d4959242d53",
    "valid.csv": "3604df83ea2ec3cec70c2eedfc7e95cc6d5cb82ccfc3b5cf565e614783e2bde2",
    "test.csv": "60e89f75e4d7d59ccfbda49ca719ce7f8e8d97c0380256d633ab240d546771ec",
    "extra_features.csv": "03c6da28bcb347982aa70a7c72b869771a6cb055654d629da924512c6fd56563",
    "daily_tickets.csv": "7deba76b112b4133a5d16b74be7e2b331ce28c7064e7fd8dd471768e5f1ba360",
    "july_tickets.csv": "10dffc093993341a361172d0aa6ea0e7c06521f140cc2a80ee90c713361f7c99",
    "july_labels.csv": "3163c57e398db6fd1c19e505548d0fa1464821ef96c77901a759abbd3a8605fa",
}
PROJECT_FILES = []
HERE = Path.cwd()
# The notebook works in its own folder ticket-model/, marked with a file, so
# it never writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("ticket-model").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

def fetch(name, url, path, expected=None):
    """Use a copy next to the notebook if there is one, or download it."""
    if path.exists() and (expected is None or sha256(path) == expected):
        return
    if (HERE / name).exists():
        shutil.copy(HERE / name, path)
    else:
        download(url, path)
    if expected and sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder ticket-model and run again.")

for name, expected in FILES.items():
    fetch(name, LABS + "data/" + name, PROJECT / "data" / name, expected)
for name in PROJECT_FILES:
    fetch(name, LABS + "project/" + name, PROJECT / name)

os.chdir(PROJECT)  # work in the project folder, as in the lesson
counts = {}
for part in ("train", "valid", "test"):
    rows = Path(f"data/{part}.csv").read_text(encoding="utf-8").splitlines()[1:]
    counts[part] = (len(rows), sum(row.endswith(",1") for row in rows))
print("Ready: train {} rows ({} escalated), valid {} ({}), test {} ({}).".format(
    *counts["train"], *counts["valid"], *counts["test"]))

## Setup: helpers

The next cell defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

It also loads pandas as `pd` and numpy as `np`, and sets pandas to show 4 decimals. Run the cell. It prints nothing.

In [ ]:
# Helpers: check an answer without showing it.
import hashlib, json, warnings
import numpy as np
import pandas as pd
pd.set_option("display.precision", 4)
pd.set_option("display.max_columns", 30)
warnings.filterwarnings("ignore", category=FutureWarning)

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. Four questions, four kinds of answer

Look at the shape of the answer that each question needs:

| Question | The answer looks like | Task |
|---|---|---|
| How many tickets will arrive tomorrow? | A number: 26, 31, 14 | **Regression** |
| Will this new ticket escalate within 72 hours? | One of a few classes: yes or no | **Classification** |
| Which customers are similar to each other? | Groups. Nobody gives the right groups. | **Clustering** |
| How many tickets escalated in April? | A count of past tickets | **No model**: a report |

Regression and classification learn from examples with the right answer, the label: that is **supervised learning**. Clustering has no label to learn from: that is **unsupervised learning**. The [lesson page](https://learning.nextia-ai.com/courses/ml/m01/choose-the-learning-task/) explains how to choose. In this notebook, you find each task in the real data.

Run the next cell. It defines `read()`, which reads a ticket file, and the names of the 10 feature columns and the label (in this course, also called the **target**).

In [ ]:
def read(path):
    """Read a ticket file. Only an empty cell is missing: "unknown" stays a category."""
    return pd.read_csv(path, parse_dates=["created_at"], dtype={"customer_id": "string"},
                       keep_default_na=False, na_values=[""])


FEATURES = ["channel", "team", "segment", "region", "priority", "order_value", "word_count",
            "customer_tenure_days", "prior_tickets_90d", "created_hour"]
TARGET = "escalated_72h"
train = read("data/train.csv")
valid = read("data/valid.csv")
print(len(FEATURES), "features;", len(train), "train tickets;", len(valid), "valid tickets")

> **Check.** You should see `10 features; 16939 train tickets; 1179 valid tickets`.

## 2. A regression target: tickets per day

In `daily_tickets.csv`, one row is one day. The column `tickets` is a number that can take many values.

> **Predict.** What is the smallest and the largest number of tickets in one day: closer to 0 and 60, or to 20 and 30? Then run the cell.

In [ ]:
daily = pd.read_csv("data/daily_tickets.csv", parse_dates=["date"])
print(daily.shape)
print(daily.head(3))
print(daily["tickets"].describe().round(2))

> **Check.** You should see `(730, 5)`, then a mean of `26.35`, a minimum of `4.0` and a maximum of `60.0`.

The label `tickets` is a number, and an answer of 27 when the truth is 26 is nearly right. That is a **regression** target. The features of one day are `weekday`, `is_weekend` and `month`. Run the next cell to see the target over time.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 3))
ax.plot(daily["date"], daily["tickets"], linewidth=0.8)
ax.set_xlabel("Day")
ax.set_ylabel("Tickets created")
ax.set_title("Tickets per day, July 2024 to June 2026")
plt.show()

> **Check.** You should see a line that jumps up and down every week (fewer tickets at weekends) and is higher each spring (the garden season).

## 3. A classification target: escalated or not

In `train.csv`, one row is one ticket. The target `escalated_72h` has only two values. Run the next cell. It splits the table into the features `X` and the target `y`, as every scikit-learn model expects.

In [ ]:
X = train[FEATURES]
y = train[TARGET]
print("X:", X.shape, " y:", y.shape)
print(y.value_counts())

> **Check.** You should see `X: (16939, 10)  y: (16939,)`, then `14956` tickets with 0 and `1983` with 1.

`y` has two classes, 0 and 1. An answer of 0.5 is not a class: the model must say yes or no. (Module 3 shows that most models first give a score.) That is a **classification** target. With two classes, it is **binary classification**.

## 4. No target: groups of customers

Grace asks a different question: "What kinds of customers do we have?" Nobody has written the right answer for any customer. Run the next cell. It makes one row per customer from the train tickets. Guests have no customer ID, so they are left out.

In [ ]:
known = train.dropna(subset=["customer_id"])
customers = known.groupby("customer_id").agg(
    tickets=("ticket_id", "size"),
    mean_words=("word_count", "mean"),
    mean_order=("order_value", "mean"),
    segment=("segment", "first"),
).round(1)
print(customers.shape)
customers.head()

> **Check.** You should see `(847, 4)` and five customers, from `C-00001` (62 tickets) to `C-00005`.

The table has features, but no label column. A **clustering** method puts similar rows into groups, for example "customers with many short tickets". Nobody can say if a group is right: a person must look at the groups and decide if they are useful. You do this in Module 7.

## 5. Answers without a model

Some questions do not need a model. Run the next cell. It shows three answers that use no machine learning:

- a **rule**: a fixed condition, for example "flag priority 1";
- a **lookup**: a table of rates from the past, for example the escalation rate of each team;
- a **report**: a count of what already happened.

In [ ]:
rule_flags = valid["priority"] == 1
print("Rule: flag priority 1 ->", rule_flags.sum(), "of", len(valid), "May tickets flagged")
print()
print("Lookup: escalation rate by team (train)")
print(train.groupby("team")[TARGET].mean().round(3).sort_values(ascending=False))
print()
april = train[train["created_at"] >= "2026-04-01"]
print("Report: April 2026 had", len(april), "tickets and", april[TARGET].sum(), "escalations")

> **Check.** You should see `163 of 1179`, then `payment` first with `0.207` and `account` last with `0.055`, then `1095` tickets and `107` escalations in April.

The report has no prediction in it: the answer is already in the data. The rule and the lookup do predict, but a person wrote them. They are fast, easy to explain and need no training. A model is worth its cost only if it beats them. In Module 2, you measure the rule.

## 6. Your turn: name the task

> **Your turn.** For each question, write `"regression"`, `"classification"`, `"clustering"` or `"no model"` in the dictionary `answers`. Ask: what does the answer look like? Is the answer already known? Then run the check cell.

1. How many minutes will the first reply to this new ticket take?
2. Will this customer write a new ticket within 30 days?
3. Which tickets are similar, so that the help pages can be grouped by topic? (Nobody has labelled the topics.)
4. How many tickets did each team get last week?
5. Is the order value of this ticket above $500?
6. How many escalations will there be next Monday?

In [ ]:
answers = {
    "q1": "regression",
    "q2": "classification",
    "q3": "clustering",
    "q4": "no model",
    "q5": "no model",
    "q6": "regression",
}

In [ ]:
allowed = {"regression", "classification", "clustering", "no model"}
if any(value not in allowed for value in answers.values()):
    print("Not yet: write one of", sorted(allowed), "for each question.")
else:
    expect_hash("the dictionary answers", answers, '1c4c57dcb3891623')

Question 5 needs no model: the order value is known, so a rule (`order_value > 500`) gives the exact answer. Question 4 is a report of the past. Questions 1 and 6 ask for a number, and question 2 for yes or no. Question 3 has no labels, so it is clustering.

## 7. Change one thing: the unit

The target `escalated_72h` is a classification target, because the unit is one ticket. Change the unit to one **day**, and the same column gives a regression target: the number of escalations per day.

> **Your turn.** Count the escalations on each day of `train`: group by the date of `created_at`, and sum `escalated_72h`. Put the result in `escalations_per_day`. Then put its mean, rounded to 2 decimals, in `mean_escalations`. Run the cell, then the check cell.

In [ ]:
escalations_per_day = train.groupby(train["created_at"].dt.date)[TARGET].sum()
mean_escalations = round(escalations_per_day.mean(), 2)
print(mean_escalations)

In [ ]:
expect_hash('mean_escalations', mean_escalations, '35dd0de5bffbe78c')

The days in train have from 0 to 10 escalations. A count per day is a number, so predicting it is regression, even though each ticket's target is yes or no. **The unit decides the task.** Choose the unit from the decision: Grace routes one ticket at a time, so her unit is the ticket.

## 8. Failure case: the wrong unit

Tomás wants a quicker way: "Label each customer as an escalator if they escalated at least once. Then send all their tickets to the senior team." The unit is now a customer, not a ticket.

> **Predict.** What share of the May tickets would his rule send to the senior team? The capacity is 20%. Then run the cell.

In [ ]:
escalations_by_customer = train.groupby("customer_id")[TARGET].sum()
escalators = escalations_by_customer[escalations_by_customer > 0].index
print("Escalators:", len(escalators), "of", train["customer_id"].nunique(), "customers")
flags = valid["customer_id"].isin(escalators)
caught = valid.loc[flags, TARGET].sum()
print(f"May tickets flagged: {flags.sum()} of {len(valid)} ({flags.mean():.1%})")
print(f"Escalations caught: {caught} of {valid[TARGET].sum()}")

> **Check.** You should see `484 of 847` customers, `952 of 1179` May tickets flagged (`80.7%`), and `163 of 183` escalations caught.

The rule catches most escalations, because it flags 4 in 5 tickets: four times the senior team's capacity. More than half of the customers escalated at least once in 22 months, so "escalator" says almost nothing about one new ticket. The question was about tickets, so the unit must be a ticket.

## 9. Module practice: name the learning and the task

This task closes Module 1. The [lesson page](https://learning.nextia-ai.com/courses/ml/m01/choose-the-learning-task/#module-practice) shows Priya's task card for Grace's question as a model.

> **Your turn.** For each question, write a pair: the kind of learning (`"supervised"`, `"unsupervised"` or `"none"`) and the task (`"regression"`, `"classification"`, `"clustering"` or `"no model"`). Ask: what does the answer look like, and do past examples have the right answer?

- **Omar:** "How many escalations will there be in each week of next month?"
- **Mei:** "Do the tickets of guest customers fall into groups that we do not know about yet?" Nobody has labelled any groups.
- **Grace:** "Which team had the most escalations in April?"

In [ ]:
module_answers = {
    "omar": ("supervised", "regression"),
    "mei": ("unsupervised", "clustering"),
    "grace": ("none", "no model"),
}

In [ ]:
learning = {"supervised", "unsupervised", "none"}
tasks = {"regression", "classification", "clustering", "no model"}
if any(a not in learning or b not in tasks for a, b in module_answers.values()):
    print("Not yet: write one kind of learning and one task for each question.")
else:
    expect_hash("the dictionary module_answers", module_answers, '727fcf629c158cb2')

Omar's question needs a number per week, and past weeks have their counts: supervised regression. Mei's question has no labels: unsupervised clustering, and a person must judge the groups. Grace's question is about the past: a report, no learning at all.

## Check your understanding and recap

Answer the **knowledge checks** on the [lesson page](https://learning.nextia-ai.com/courses/ml/m01/choose-the-learning-task/#knowledge-check). They count toward your certificate when you are signed in and enrolled.

In this lesson, you found three tasks in the real data: tickets per day (regression), escalated or not (classification) and customers without a label (clustering). You also saw three answers without a model: a rule, a lookup and a report. The unit decides the task: the same column gives a classification target per ticket and a regression target per day. This is the end of Module 1. Next, in Module 2, you turn Grace's need into an exact prediction problem.

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Translate a need](https://learning.nextia-ai.com/courses/ml/m02/translate-a-need/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/ml/m01/choose-the-learning-task/)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.